In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.apm_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.utils.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'DET': ['Jalen Duren']}

Out Players:
{'DET': ['Kevin Huerter'], 'ORL': ['Franz Wagner', 'Jonathan Isaac'], 'TOR': ['Brandon Ingram', 'Immanuel Quickley'], 'LAL': ['Luka Doncic'], 'HOU': ['Kevin Durant', 'Fred VanVleet']}
Note: DET (Pistons) has 4 confirmed players - lineup will still be updated
Successfully updated C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\src\utils\team_info.py
Updated 6 teams with confirmed lineups
Updated 1 teams with questionable players


### Dataset

In [4]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,name
27518,NaN,NaN,868,2025-26,1629130,Duncan Robinson,Duncan,1610612765,DET,Detroit Pistons,42500105,2026-04-29,DET vs. ORL,W,24.953333,4,8,0.500,3,6,0.500,1,2,0.500,1,2,3,1,1,0,0,0,1,3,12,2,16.1,0,0,19.0,1,24:57,1,105.9,103.8,103.8,102.2,100.0,100.0,3.7,3.8,3.8,0.067,1.00,9.1,0.045,0.069,0.059,9.1,9.2,0.688,0.676,0.169,0.171,99.80,101.95,84.96,101.95,0.096,53,4.0,8.0,G,4.48,2.00,2.0,4.0,5.0,30.0,0.0,0.0,19.0,0.0,2.0,0.000,4.0,6.0,0.667,0.0,0.0,0.0,39,80,0.488,10,28,0.357,28,35,0.800,16,33,49,20,17.0,10,5,5,21,26,116,7.0,120.3,120.8,107.7,111.2,12.6,9.6,0.513,1.18,15.0,0.386,0.700,0.553,0.177,0.550,0.608,98.8,97.0,80.83,96,0.578,1610612753,ORL,Orlando Magic,38,80,0.475,17,38,0.447,16,30,0.533,8,25,33,21,16.0,12,5,5,26,21,109,-7.0,107.7,111.2,120.3,120.8,-12.6,-9.6,0.553,1.31,15.7,0.300,0.614,0.447,0.163,0.581,0.585,98.8,97.0,80.83,98,0.422,1,SF,31.0,NaN,NaN,-11.0,211.0,1,0.480898,0.040075,0.120224,1,3,Cade Cunningham,Jalen Duren,Paul Reed,0,0,2,1,Duncan Robinson
27519,NaN,NaN,869,2025-26,1628386,Jarrett Allen,Jarrett,1610612739,CLE,Cleveland Cavaliers,42500135,2026-04-29,CLE vs. TOR,W,25.106667,4,5,0.800,0,0,0.000,1,2,0.500,0,3,3,1,3,0,3,0,0,1,9,-4,20.1,0,0,19.0,1,25:06,1,113.1,114.3,114.3,122.8,125.9,125.9,-9.8,-11.6,-11.6,0.053,0.33,10.0,0.000,0.115,0.067,30.0,30.4,0.800,0.765,0.153,0.154,107.02,105.15,87.63,105.15,0.079,56,4.0,5.0,C,4.21,1.89,3.0,8.0,10.0,21.0,0.0,0.0,12.0,3.0,4.0,0.750,1.0,1.0,1.000,2.0,4.0,0.5,43,81,0.531,18,36,0.500,21,28,0.750,4,31,35,20,15.0,8,8,8,16,21,125,5.0,119.8,122.5,113.2,118.8,6.6,3.7,0.465,1.33,15.4,0.214,0.600,0.433,0.147,0.642,0.670,105.2,101.5,84.58,102,0.510,1610612761,TOR,Toronto Raptors,44,95,0.463,15,38,0.395,17,25,0.680,15,33,48,32,15.0,8,8,8,21,16,120,-5.0,113.2,118.8,119.8,122.5,-6.6,-3.7,0.727,2.13,20.8,0.400,0.786,0.567,0.149,0.542,0.566,105.2,101.5,84.58,101,0.490,1,C,27.0,NaN,NaN,-9.5,219.5,1,0.358471,0.039830,0.119490,1,0,Donovan Mitchell,James Harden,Jarrett Allen,1,0,3,1,Jarrett Allen
27520,NaN,NaN,845,2025-26,1629060,Rui Hachimu

### Load latest odds on file

In [5]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260501_172119.json


,home_team,away_team,commence_time,bookmakers
0,Orlando Magic,Detroit Pistons,2026-05-01 23:09:16+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Toronto Raptors,Cleveland Cavaliers,2026-05-01 23:41:07+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Houston Rockets,Los Angeles Lakers,2026-05-02 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Boston Celtics,Philadelphia 76ers,2026-05-02 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,San Antonio Spurs,Minnesota Timberwolves,2026-05-05 01:30:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [6]:
scenarios = player_scenarios(base_df, 'Desmond Bane', 'MIN')
result = get_game_context(base_df, 'Desmond Bane', team_odds)
result

{'player': 'Desmond Bane',
 'team': 'Orlando Magic',
 'opponent': 'Detroit Pistons',
 'is_home': True,
 'commence_time': Timestamp('2026-05-01 23:09:16+0000', tz='UTC'),
 'bookmaker': 'DraftKings',
 'active_stars': 2,
 'active_star_names': ['Paolo Banchero', 'Desmond Bane'],
 'spread': -16.5,
 'spread_price': -110,
 'total': 199.5,
 'total_over_price': -120,
 'total_under_price': -110}

In [15]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-01 17:19:41
US latest pull: 2026-05-01 17:21:19


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Desmond Bane,Over,23.5,-137,2026-05-01,2026-05-02T00:18:46Z,2026-05-01 17:19:41
1,PrizePicks,player_points,Desmond Bane,Under,23.5,-137,2026-05-01,2026-05-02T00:18:46Z,2026-05-01 17:19:41
2,PrizePicks,player_points,Paolo Banchero,Over,19.5,-137,2026-05-01,2026-05-02T00:18:46Z,2026-05-01 17:19:41
3,PrizePicks,player_points,Paolo Banchero,Under,19.5,-137,2026-05-01,2026-05-02T00:18:46Z,2026-05-01 17:19:41
4,PrizePicks,player_points,R.J. Barrett,Over,27.5,-137,2026-05-01,2026-05-02T00:19:11Z,2026-05-01 17:19:41


### Load my models

In [16]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-02.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [17]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Desmond Bane,PTS,27.80,35.93,40.83,0.3074,0.5291,0.7898,8.54,19.01,32.25,"[0.6616166458912647, 0.6411062225015713, 0.562..."
1,Paolo Banchero,PTS,31.58,39.85,43.97,0.3788,0.5918,0.9380,11.96,23.58,41.24,"[0.7968127490039841, 0.3006681514476614, 0.511..."
2,Scottie Barnes,PTS,29.81,37.49,41.95,0.3259,0.5451,0.8267,9.72,20.43,34.68,"[0.6304248515303793, 0.8108108108108107, 0.180..."
3,Evan Mobley,PTS,24.88,33.42,38.74,0.2961,0.5098,0.7515,7.37,17.04,29.12,"[0.5914567360350492, 0.5845254576219043, 0.261..."
4,Collin Murray-Boyles,PTS,17.56,23.30,30.38,0.1696,0.4587,0.7398,2.98,10.69,22.47,"[0.5266237565827969, 0.4213483146067416, 0.738..."
5,Sam Merrill,PTS,17.65,22.29,28.63,0.1798,0.4553,0.7374,3.17,10.15,21.11,"[0.6064209274673009, 0.4164442071542979, 0.467..."
6,Jakob Poeltl,PTS,13.95,20.10,27.10,0.1768,0.4412,0.7219,2.47,8.87,19.56,"[0.6091027014832779, 0.4391217564870259, 0.553..."
7,Austin Reaves,PTS,29.01,38.00,42.16,0.3391,0.5751,0.8806,9.84,21.86,37.13,"[0.6440532417346501, 0.8058925476603119, 0.778..."
8,Alperen Sengun,PTS,29.88,38.23,43.92,0.3245,0.5422,0.8113,9.70,20.73,35.63,"[0.7003891050583657, 0.426179604261796, 1.1330..."
9,LeBron James,PTS,29.21,36.95,42.54,0.3343,0.5552,0.8203,9.76,20.52,34.89,"[0.3098106712564544, 0.6673114119922631, 0.373..."


In [18]:
from src.live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
ast_preds

Desmond Bane [PTS]  MIN: 35.9→36.2 (Δ+0.50)  RATE: 0.5291→0.5273 (Δ-0.0035)
Paolo Banchero [PTS]  MIN: 39.9→40.3 (Δ+0.95)  RATE: 0.5918→0.6092 (Δ+0.0349)
Scottie Barnes [PTS]  MIN: 37.5→39.3 (Δ+3.66)  RATE: 0.5451→0.5279 (Δ-0.0344)
Evan Mobley [PTS]  MIN: 33.4→33.0 (Δ-0.92)  RATE: 0.5098→0.5064 (Δ-0.0069)
Collin Murray-Boyles [PTS]  MIN: 23.3→25.3 (Δ+4.00)  RATE: 0.4587→0.4439 (Δ-0.0296)
Sam Merrill [PTS]  MIN: 22.3→20.3 (Δ-4.00)  RATE: 0.4553→0.4353 (Δ-0.0400)
Jakob Poeltl [PTS]  MIN: 20.1→21.6 (Δ+3.02)  RATE: 0.4412→0.4612 (Δ+0.0400)
Austin Reaves [PTS]  MIN: 38.0→38.3 (Δ+0.66)  RATE: 0.5751→0.5551 (Δ-0.0400)
Alperen Sengun [PTS]  MIN: 38.2→38.6 (Δ+0.65)  RATE: 0.5422→0.5246 (Δ-0.0353)
LeBron James [PTS]  MIN: 37.0→36.9 (Δ-0.19)  RATE: 0.5552→0.5352 (Δ-0.0400)
Amen Thompson [PTS]  MIN: 41.1→41.8 (Δ+1.39)  RATE: 0.4727→0.4767 (Δ+0.0080)
Jabari Smith Jr [PTS]  MIN: 30.4→32.1 (Δ+3.40)  RATE: 0.4454→0.4654 (Δ+0.0400)
Reed Sheppard [PTS]  MIN: 30.2→31.7 (Δ+3.02)  RATE: 0.4417→0.4617 (Δ+0.

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Ausar Thompson,AST,22.58,30.77,37.77,0.0334,0.0901,0.1531,0.75,2.77,5.78,"[0.0, 0.0, 0.163579, 0.161204, 0.244965, 0.067..."
1,Jalen Suggs,AST,26.41,35.26,40.88,0.0747,0.1402,0.2217,1.97,4.94,9.06,"[0.114174, 0.187378, 0.180336, 0.336414, 0.279..."
2,Paolo Banchero,AST,31.96,40.33,44.49,0.0728,0.1429,0.2325,2.33,5.76,10.34,"[0.189504, 0.102153, 0.21949, 0.116265, 0.1638..."
3,James Harden,AST,29.84,39.11,42.98,0.0901,0.1736,0.2809,2.69,6.79,12.07,"[0.28879, 0.197258, 0.172504, 0.497046, 0.3793..."
4,LeBron James,AST,29.13,36.85,42.43,0.0949,0.1677,0.2885,2.76,6.18,12.24,"[0.251796, 0.25467, 0.208155, 0.354293, 0.1910..."
5,Austin Reaves,AST,29.26,38.33,42.53,0.0533,0.1267,0.1924,1.56,4.85,8.18,"[0.120398, 0.194389, 0.169723, 0.123201, 0.116..."
6,Alperen Sengun,AST,30.13,38.56,44.29,0.0689,0.1432,0.2420,2.08,5.52,10.72,"[0.066138, 0.20122, 0.20805, 0.315643, 0.12813..."
7,Amen Thompson,AST,32.59,41.83,46.03,0.0822,0.1458,0.2333,2.68,6.10,10.74,"[0.236667, 0.081033, 0.152783, 0.219732, 0.160..."
8,Joel Embiid,AST,24.51,31.05,38.68,0.0623,0.1298,0.2233,1.53,4.03,8.64,"[0.169703, 0.220224, 0.101349, 0.092008, 0.179..."
9,Tyrese Maxey,AST,32.27,41.68,46.44,0.0796,0.1530,0.2458,2.57,6.38,11.41,"[0.132763, 0.186941, 0.205943, 0.265879, 0.230..."


### Get Line Probabilities

In [19]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
51,Payton Pritchard,REB,3.0,22.43,29.40,34.96,1.07,3.60,8.63,0.611,0.389
29,Anthony Black,REB,2.5,18.62,25.77,32.94,0.71,2.99,7.34,0.695,0.305
74,Scottie Barnes,PTS,26.5,31.26,39.32,44.00,9.87,20.76,35.23,0.269,0.731
125,Jamal Cain,PTS,5.5,16.00,23.59,30.77,2.00,9.24,21.57,0.715,0.285
56,Jaden McDaniels,REB,4.5,25.12,35.14,40.44,1.89,5.75,12.11,0.658,0.342
129,Jamal Shead,PTS,9.5,22.65,32.05,39.93,2.66,12.62,28.13,0.531,0.469
40,Rui Hachimura,REB,3.5,25.24,33.38,39.24,1.17,3.93,9.24,0.493,0.507
100,Quentin Grimes,PTS,6.5,19.15,25.69,33.17,3.69,11.72,24.55,0.784,0.216
41,Marcus Smart,REB,2.5,24.33,33.92,41.29,0.72,2.79,7.04,0.492,0.508
111,Julian Champagnie,PTS,8.5,22.22,29.93,37.06,3.38,12.92,25.91,0.735,0.265


In [20]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
92,Jaylen Brown,PTS,25.5,25.92,30.45,39.31,10.37,18.86,36.70,0.366,0.634,PTS,Underdog,Philadelphia 76ers,-7.5,205.5,114.4,17.0,100.40,15.0,-105.0,-114.0,0.512,0.533,28.2,26.0,7.33,2.7,0.5,-0.368,0.644,0.356,25.73,-33.17,0.4,0.6,0.73,0.49,35.52,5.22,0.32,0.03,25.08,12.0
85,Tari Eason,PTS,13.5,23.12,33.68,41.66,4.40,14.87,30.64,0.664,0.336,PTS,Underdog,Los Angeles Lakers,-3.5,204.0,115.5,20.0,99.22,22.0,102.0,-110.0,0.495,0.524,11.8,13.5,7.35,-1.7,0.0,0.231,0.409,0.591,-17.38,12.83,0.6,0.5,0.60,0.33,28.32,7.19,0.17,0.07,12.22,9.0
97,Derrick White,PTS,11.5,23.32,27.71,35.82,3.99,11.96,26.31,0.319,0.681,PTS,Underdog,Philadelphia 76ers,-7.5,205.5,114.4,17.0,100.40,15.0,100.0,-122.0,0.500,0.550,9.7,9.5,3.23,-1.8,-2.0,0.557,0.289,0.711,-42.20,29.38,0.0,0.2,0.27,0.75,33.05,6.46,0.15,0.06,14.31,13.0
107,De'Aaron Fox,PTS,17.5,27.20,35.63,40.93,9.56,19.35,34.36,0.709,0.291,PTS,Underdog,Minnesota Timberwolves,-14.0,216.5,112.5,8.0,101.50,10.0,-113.0,-108.0,0.531,0.519,19.5,18.0,4.88,2.0,0.5,-0.410,0.659,0.341,24.22,-34.33,0.6,0.6,0.47,0.65,33.91,4.33,0.25,0.03,26.43,7.0
83,Jabari Smith Jr,PTS,17.5,28.00,32.15,42.47,6.67,14.96,32.68,0.497,0.503,PTS,Underdog,Los Angeles Lakers,-3.5,204.0,115.5,20.0,99.22,22.0,-105.0,-112.0,0.512,0.528,19.2,18.5,2.97,1.7,1.0,-0.572,0.716,0.284,39.79,-46.24,0.6,0.7,0.60,0.24,37.77,6.18,0.21,0.04,18.56,9.0


In [21]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
87,Rui Hachimura,PTS,12.5,25.24,33.38,39.24,3.82,13.98,27.17,0.593,0.407,PTS,PrizePicks,Houston Rockets,3.5,204.0,112.1,6.0,96.98,29.0,-137.0,-137.0,0.578,0.578,15.7,13.5,4.22,3.7,1.5,-0.877,0.810,0.190,40.12,-67.13,0.8,0.8,0.67,0.50,34.81,6.64,0.17,0.03,11.27,11.0
22,Marcus Smart,AST,3.0,24.33,33.92,41.29,1.09,3.38,7.21,0.693,0.307,AST,PrizePicks,Houston Rockets,3.5,204.0,112.1,6.0,96.98,29.0,-137.0,-137.0,0.578,0.578,5.8,6.0,3.12,2.8,3.0,-0.897,0.815,0.185,40.99,-68.00,0.8,0.8,0.60,0.43,31.22,6.13,0.18,0.06,4.44,9.0
112,Keldon Johnson,PTS,8.5,15.80,19.65,25.79,3.48,9.04,18.93,0.622,0.378,PTS,PrizePicks,Minnesota Timberwolves,-14.0,216.5,112.5,8.0,101.50,10.0,100.0,-106.0,0.500,0.515,10.9,9.5,5.84,2.4,1.0,-0.411,0.659,0.341,31.80,-33.73,0.2,0.6,0.73,0.72,21.12,3.89,0.22,0.06,18.00,6.0
110,Dylan Harper,PTS,9.5,19.19,24.70,30.94,3.82,10.90,21.81,0.700,0.300,PTS,PrizePicks,Minnesota Timberwolves,-14.0,216.5,112.5,8.0,101.50,10.0,-104.0,-109.0,0.510,0.522,12.7,12.5,6.68,3.2,3.0,-0.479,0.684,0.316,34.17,-39.41,0.6,0.7,0.80,0.64,25.18,4.25,0.19,0.03,8.67,3.0
46,VJ Edgecombe,REB,5.5,33.19,41.58,45.30,2.24,5.86,10.52,0.646,0.354,REB,PrizePicks,Boston Celtics,7.5,205.5,111.7,4.0,95.58,30.0,-114.0,105.0,0.533,0.488,7.0,7.0,2.36,1.5,1.5,-0.636,0.738,0.262,38.54,-46.29,0.6,0.8,0.60,0.55,37.39,3.42,0.20,0.06,6.00,9.0


In [23]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
94,Jayson Tatum,PTS,23.5,25.28,29.32,38.71,9.30,16.94,34.61,0.307,0.693,PTS,Betr DFS,Philadelphia 76ers,-7.5,205.5,114.4,17.0,100.40,15.0,-123.0,106.0,0.552,0.485,24.1,24.0,2.73,0.6,0.5,-0.220,0.587,0.413,6.42,-14.92,0.8,0.6,0.53,0.59,37.03,3.75,0.27,0.04,26.67,9.0
116,Ausar Thompson,PTS,8.5,22.58,30.77,37.77,2.77,12.02,25.40,0.597,0.403,PTS,Betr DFS,Orlando Magic,16.5,199.2,113.6,13.0,100.56,14.0,-137.0,-137.0,0.578,0.578,9.9,9.0,3.63,1.4,0.5,-0.386,0.650,0.350,12.45,-39.45,0.4,0.5,0.40,0.55,28.56,5.35,0.16,0.05,9.73,11.0
85,Tari Eason,PTS,13.5,23.12,33.68,41.66,4.40,14.87,30.64,0.664,0.336,PTS,Betr DFS,Los Angeles Lakers,-3.5,204.0,115.5,20.0,99.22,22.0,102.0,-110.0,0.495,0.524,11.8,13.5,7.35,-1.7,0.0,0.231,0.409,0.591,-17.38,12.83,0.6,0.5,0.60,0.33,28.32,7.19,0.17,0.07,12.22,9.0
0,Ausar Thompson,AST,3.5,22.58,30.77,37.77,0.75,2.77,5.78,0.429,0.571,AST,Betr DFS,Orlando Magic,16.5,199.2,113.6,13.0,100.56,14.0,135.0,-160.0,0.426,0.615,3.4,3.5,2.12,-0.1,0.0,0.047,0.481,0.519,13.03,-15.66,0.2,0.5,0.53,0.31,28.56,5.35,0.16,0.05,2.82,11.0
93,Tyrese Maxey,PTS,23.5,32.27,41.68,46.44,11.33,23.97,42.26,0.642,0.358,PTS,Betr DFS,Boston Celtics,7.5,205.5,111.7,4.0,95.58,30.0,-115.0,-105.0,0.535,0.512,24.3,23.5,5.19,0.8,0.0,-0.154,0.561,0.439,4.88,-14.29,0.6,0.5,0.53,0.69,37.30,4.89,0.28,0.05,27.58,12.0


In [24]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
82,Amen Thompson,PTS,18.5,32.59,41.83,46.03,8.22,19.94,34.99,0.570,0.430,PTS,DraftKings Pick6,Los Angeles Lakers,-3.5,204.0,115.5,20.0,99.22,22.0,100.0,105.0,0.500,0.488,21.8,20.0,7.55,3.3,1.5,-0.437,0.669,0.331,33.80,-32.15,0.4,0.6,0.40,0.38,41.04,4.68,0.21,0.03,21.10,10.0
87,Rui Hachimura,PTS,12.5,25.24,33.38,39.24,3.82,13.98,27.17,0.593,0.407,PTS,DraftKings Pick6,Houston Rockets,3.5,204.0,112.1,6.0,96.98,29.0,108.0,-128.0,0.481,0.561,15.7,13.5,4.22,3.2,1.0,-0.758,0.776,0.224,61.41,-60.10,0.8,0.8,0.67,0.50,34.81,6.64,0.17,0.03,11.27,11.0
77,Sam Merrill,PTS,5.5,16.07,20.29,26.06,2.76,8.83,18.37,0.734,0.266,PTS,DraftKings Pick6,Toronto Raptors,1.0,224.8,112.1,5.0,99.22,21.0,104.0,-115.0,0.490,0.535,8.7,8.0,5.48,3.2,2.5,-0.584,0.720,0.280,46.88,-47.65,0.6,0.7,0.80,0.68,23.91,4.19,0.14,0.06,7.80,10.0
68,Evan Mobley,REB,10.5,24.54,32.96,38.20,3.67,7.74,13.79,0.248,0.752,REB,DraftKings Pick6,Toronto Raptors,1.0,224.8,112.1,5.0,99.22,21.0,-105.0,-108.0,0.512,0.519,8.4,7.5,4.14,-2.1,-3.0,0.507,0.306,0.694,-40.26,33.66,0.0,0.1,0.13,0.32,30.21,5.18,0.21,0.06,8.92,12.0
15,Duncan Robinson,AST,2.5,19.44,25.89,32.45,0.18,1.53,4.23,0.338,0.662,AST,DraftKings Pick6,Orlando Magic,16.5,199.2,113.6,13.0,100.56,14.0,-120.0,-110.0,0.545,0.524,2.2,2.5,1.23,-0.3,0.0,0.244,0.404,0.596,-25.93,13.78,0.6,0.5,0.40,0.36,26.36,3.02,0.17,0.05,2.25,12.0


In [25]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
115,Jalen Duren,PTS,10.5,19.84,26.08,32.88,5.74,12.57,26.01,0.618,0.382,PTS,Betr DFS,Orlando Magic,16.5,199.2,113.6,13.0,100.56,14.0,105.0,-130.0,0.488,0.565,14.8,14.0,5.29,4.3,3.5,-0.813,0.792,0.208,62.36,-63.20,0.6,0.8,0.80,0.74,29.71,3.77,0.20,0.05,13.08,12.0
10,Jaylen Brown,AST,4.0,25.92,30.45,39.31,1.27,3.65,7.47,0.352,0.648,AST,Betr DFS,Philadelphia 76ers,-7.5,205.5,114.4,17.0,100.40,15.0,-137.0,-137.0,0.578,0.578,3.5,3.5,1.78,-0.5,-0.5,0.281,0.389,0.611,-32.71,5.70,0.2,0.2,0.40,0.46,35.52,5.22,0.32,0.03,4.67,12.0
49,Tyrese Maxey,REB,4.0,32.27,41.68,46.44,1.42,4.81,9.83,0.535,0.465,REB,PrizePicks,Boston Celtics,7.5,205.5,111.7,4.0,95.58,30.0,-137.0,-137.0,0.578,0.578,3.8,3.0,2.97,-0.2,-1.0,0.067,0.473,0.527,-18.17,-8.83,0.2,0.2,0.33,0.28,37.30,4.89,0.28,0.05,4.25,12.0
41,Marcus Smart,REB,2.5,24.33,33.92,41.29,0.72,2.79,7.04,0.492,0.508,REB,Underdog,Houston Rockets,3.5,204.0,112.1,6.0,96.98,29.0,-117.0,105.0,0.539,0.488,2.8,2.0,1.40,0.3,-0.5,-0.214,0.585,0.415,8.50,-14.93,0.4,0.4,0.47,0.49,31.22,6.13,0.18,0.06,2.33,9.0
5,Austin Reaves,AST,5.5,29.26,38.33,42.53,1.56,4.85,8.18,0.405,0.595,AST,Betr DFS,Houston Rockets,3.5,204.0,112.1,6.0,96.98,29.0,-112.0,100.0,0.528,0.500,5.5,5.0,2.37,0.0,-0.5,0.000,0.500,0.500,-5.36,0.00,0.4,0.4,0.47,0.47,36.05,5.19,0.24,0.06,5.57,7.0


### Get top EVs for 2 legs

In [26]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 78  |  Pairs: 103  |  Slate: 5  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json


In [27]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 28  |  Pairs: 2  |  Slate: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json


In [28]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 42  |  Pairs: 74  |  Slate: 5  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json


In [29]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 51  |  Pairs: 61  |  Slate: 4  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json


### Top EVs for 3 Legs

In [30]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 78  |  Triples: 1386  |  Slate: 5  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 46  |  Triples: 52  |  Slate: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json


In [31]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 51  |  Triples: 480  |  Slate: 3  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json


In [32]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 42  |  Triples: 595  |  Slate: 4  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
